#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT / "library"))

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# set device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# set confounders
confounders = ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10','f11']
input_dim = len(confounders)

#### helpers

In [ ]:
def auuc_sep_rel_prop1(y, t, u):
    """
    Compute separate relative AUUC using Prop. 1 of https://dl.acm.org/doi/abs/10.1145/3447548.3467395
    """
    y_t = y[t==1]
    y_c = y[t==0]
    
    u_t = u[t==1]
    u_c = u[t==0]
    
    auc_t = roc_auc_score(y_t, u_t)
    auc_c = roc_auc_score(y_c, u_c)
    
    lambda_t = np.mean(y_t)*(1 - np.mean(y_t))
    lambda_c = np.mean(y_c)*(1 - np.mean(y_c))
    
    return lambda_t*auc_t - lambda_c*auc_c + (np.mean(y_t)**2) / 2. - (np.mean(y_c)**2) / 2.

In [ ]:
def load_checkpoint(model_cls, path, input_dim, device, hidden_dims=(128, 64)):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {path}")

    last_error = None

    for hidden_dim in hidden_dims:
        try:
            model = model_cls(input_dim=input_dim, hidden_dim=hidden_dim).to(device)
            model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
            model.eval()
            return model
        except RuntimeError as err:
            last_error = err

    raise RuntimeError(f"Could not load checkpoint with hidden dims {hidden_dims}: {path}") from last_error

In [ ]:
def load_models_criteo(seed, confounders, device):
    input_dim = len(confounders)

    base_dir = ROOT / "experiments" / "criteo"

    ranker_dir = base_dir / "chkpts" / "rankers" / f"seed_{seed}"
    pointwise_dir = base_dir / "chkpts" / "pointwise" / f"seed_{seed}"
    nuisance_dir = base_dir / "chkpts" / "nuisances" / f"seed_{seed}"

    rank_learner = load_checkpoint(
        ClassificationHead,
        ranker_dir / "orthogonal.pt",
        input_dim,
        device,
    )

    pi_model = load_checkpoint(
        ClassificationHead,
        ranker_dir / "plug_in.pt",
        input_dim,
        device,
    )

    dr_model = load_checkpoint(
        RegressionHead,
        pointwise_dir / "cate_model.pt",
        input_dim,
        device,
    )

    m0_model = load_checkpoint(
        ClassificationHead,
        nuisance_dir / "mu0_model.pt",
        input_dim,
        device,
    )

    m1_model = load_checkpoint(
        ClassificationHead,
        nuisance_dir / "mu1_model.pt",
        input_dim,
        device,
    )

    return rank_learner, pi_model, dr_model, m0_model, m1_model

In [ ]:
def get_estimates_criteo(rank_learner, pi_model, dr_model, m0_model, m1_model, test_loader, test_df, device):
    for model in [rank_learner, pi_model, dr_model, m0_model, m1_model]:
        model.eval()

    estimates = {
        "DR_score": [],
        "T_score": [],
        "rank_learner_score": [],
        "plugin_score": []}

    with torch.inference_mode():
        for x, _, _, _ in test_loader:
            x = x.to(device)

            estimates["DR_score"].append(dr_model(x).squeeze(-1).cpu())

            m0_hat = torch.sigmoid(m0_model(x)).squeeze(-1)
            m1_hat = torch.sigmoid(m1_model(x)).squeeze(-1)
            estimates["T_score"].append((m1_hat - m0_hat).cpu())

            estimates["rank_learner_score"].append(rank_learner(x).squeeze(-1).cpu())
            estimates["plugin_score"].append(pi_model(x).squeeze(-1).cpu())

    # add estimates to df
    eval_df = test_df.copy()
    for name, parts in estimates.items():
        eval_df[name] = torch.cat(parts).numpy()

    return eval_df

In [ ]:
def compute_metrics_criteo(eval_df):
    y = eval_df["Y"].to_numpy()
    t = eval_df["T"].to_numpy()

    specs = [
        ("DR", "DR_score"),
        ("T", "T_score"),
        ("rank_learner", "rank_learner_score"),
        ("plug_in", "plugin_score")]

    rows = []

    for name, score_col in specs:
        rows.append({
            "model": name,
            "auuc": auuc_sep_rel_prop1(y=y, t=t, u=eval_df[score_col].to_numpy()),})

    return pd.DataFrame(rows)

#### evaluation

In [ ]:
# init collector
all_metrics = []

# loop over test sizes and seeds
for test_size in [50_000, 500_000, 1_000_000]:
    for seed in range(5):
    
        # track progress
        print(f" -> Seed {seed}, test size {test_size}")
        set_seed(seed)
    
        # get testing data
        test_df = pd.read_csv(ROOT / "experiments" / "criteo" / "data" / "datasets" / f"seed_{seed}" / f"test_{test_size}.csv")
        test_df["cate"] = 0; test_df["M0"] = 0; test_df["M1"] = 0
        test_loader = DataLoader(EvalDataset(test_df, confounders), batch_size=1024, shuffle=False)

        # load models
        rank_learner, pi_model, dr_model, m0_model, m1_model = load_models_criteo(seed=seed, confounders=confounders, device=device)

        # get estimates
        eval_df = get_estimates_criteo(rank_learner, pi_model, dr_model, m0_model, m1_model, test_loader, test_df, device)

        # compute metrics
        df_metrics = compute_metrics_criteo(eval_df)
        df_metrics["seed"] = seed
        df_metrics["test_size"] = test_size
        all_metrics.append(df_metrics)

# summarize
df_all = pd.concat(all_metrics, ignore_index=True)
df_summary = (df_all.groupby(["model", "test_size"])["auuc"].agg(["mean", "std"]).reset_index())